# JetBot 資料收集（Gamepad）

透過遊戲手把（Gamepad）標記目標座標，收集道路跟隨訓練資料。

## 流程
1. 啟動 JetBot CSI 相機（224×224）
2. 使用 Gamepad 搖桿設定目標座標 (X, Y)
3. 按下按鈕儲存影像（座標編碼於檔名）
4. 建議收集 **50～200 張**影像

## 1. 匯入套件

In [ ]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg
import os
import uuid
import datetime
import numpy as np
import cv2

## 2. 啟動相機 & 建立顯示元件

In [ ]:
# 啟動 CSI 相機 (224x224)
camera = Camera.instance(width=224, height=224)

# 建立即時影像顯示 widget
image_widget = widgets.Image(format='jpeg', width=224, height=224)

# 將相機畫面綁定到 widget
camera_link = traitlets.dlink(
    (camera, 'value'),
    (image_widget, 'value'),
    transform=bgr8_to_jpeg
)

## 3. 建立座標滑桿 & Gamepad 控制

In [ ]:
# 建立 X, Y 滑桿（範圍 -1 到 1）
# 當 x_slider = -1 時，目標點在畫面最左邊
# 當 x_slider = 1 時，目標點在畫面最右邊
x_slider = widgets.FloatSlider(
    min=-1.0, max=1.0, step=0.001,
    description='x', value=0
)
y_slider = widgets.FloatSlider(
    min=-1.0, max=1.0, step=0.001,
    description='y', value=0
)

# 連接 Gamepad
controller = widgets.Controller(index=0)
display(controller)

## 4. 設定 Gamepad 搖桿軸對應

In [ ]:
# 將 Gamepad 搖桿軸綁定到 x, y 滑桿
# axis[0] = 左搖桿水平軸 → X 座標
# axis[3] = 右搖桿垂直軸 → Y 座標（依手把型號調整）
x_link = traitlets.dlink(
    (controller.axes[0], 'value'),
    (x_slider, 'value')
)
y_link = traitlets.dlink(
    (controller.axes[3], 'value'),
    (y_slider, 'value')
)

## 5. 建立儲存目錄 & 計數器

In [ ]:
DATASET_DIR = 'dataset_xy'
os.makedirs(DATASET_DIR, exist_ok=True)

# 計數已收集的影像數量
count_widget = widgets.IntText(
    description='count',
    value=len(os.listdir(DATASET_DIR))
)

def save_snapshot(change):
    """當按鈕按下時，儲存帶有座標的影像"""
    if change['new']:
        x = x_slider.value
        y = y_slider.value
        
        # 將 [-1, 1] 座標縮放到整數（方便檔名儲存）
        x_int = int(x * 50 + 50)
        y_int = int(y * 50 + 50)
        
        # 格式：xy_XXX_YYY_uuid.jpg
        filename = 'xy_%03d_%03d_%s.jpg' % (
            x_int, y_int, uuid.uuid1()
        )
        image_path = os.path.join(DATASET_DIR, filename)
        
        # 儲存當前相機影像
        with open(image_path, 'wb') as f:
            f.write(bgr8_to_jpeg(camera.value))
        
        count_widget.value = len(os.listdir(DATASET_DIR))

# 綁定 Gamepad 按鈕（button[0] 通常是 A 鍵）
controller.buttons[0].observe(save_snapshot, names='value')

print(f'儲存目錄: {DATASET_DIR}')
print(f'目前已收集: {count_widget.value} 張')

## 6. 顯示即時畫面 & 座標標記

In [ ]:
def display_xy(camera_image):
    """在影像上繪製目標座標綠點"""
    image = np.copy(camera_image)
    x = x_slider.value
    y = y_slider.value
    
    # 將 [-1, 1] 轉換為像素座標
    # x_slider = -1 時，pixel_x = -1 * 224/2 + 112 = 0
    # x_slider = 1  時，pixel_x =  1 * 224/2 + 112 = 224
    pixel_x = int(x * 224 / 2 + 112)
    pixel_y = int(y * 224 / 2 + 112)
    
    # 繪製綠色目標點
    image = cv2.circle(image, (pixel_x, pixel_y), 8, (0, 255, 0), 3)
    
    return bgr8_to_jpeg(image)

# 使用帶標記的影像更新顯示
camera_link.unlink()
camera_link = traitlets.dlink(
    (camera, 'value'),
    (image_widget, 'value'),
    transform=display_xy
)

# 顯示所有控制元件
display(
    widgets.HBox([image_widget]),
    x_slider,
    y_slider,
    count_widget
)

print('\n操作說明：')
print('1. 用搖桿移動綠色目標點到 JetBot 應該前進的位置')
print('2. 按下 A 鍵（button[0]）儲存當前影像')
print('3. 建議收集 50~200 張影像')

## 7. 停止相機（切換 Notebook 前務必執行）

In [ ]:
camera.stop()
print('相機已停止 ✓')